# Billionaire Granny — Arabic multi-speaker dubbing

This notebook runs the resumable dubbing pipeline inside a real Google Colab runtime and uses your mounted Google Drive for the source and checkpoints.

## What it does
- Detects source speech and timestamps with faster-whisper.
- Clusters likely speaker turns and assigns separate synthetic voice profiles.
- Translates each segment to Arabic, applies supported expressive tags, and synthesizes voice lines.
- Saves resumable segment/model checkpoints in the Drive output folder and reuses verified checkpoints after an interrupted run.
- Muxes the Arabic audio into an MP4, validates the streams and full decode, hashes the final file, and saves a quality report plus Arabic subtitles in Drive.

**Memory guard:** the run stops before loading large models if the runtime exposes less than 16 GiB total RAM, less than 10 GiB currently available, or less than 16 GiB free local disk. Select a **High-RAM** runtime before running all cells. The notebook will not silently fall back to a smaller RAM configuration.

**Important limitation:** the existing pipeline aligns speech to timestamps but does not animate mouth movement or track the active visible speaker. The report will honestly mark visual lip-sync as *not performed*; this is not advertised as Netflix-certified quality. A review of translations, voice choices and final video is still advisable.

Use this only for content you have the right to dub and store.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil
import subprocess
import json

SOURCE = Path('/content/drive/MyDrive/دبلجة/Billionaire Granny Takes Home Fainted Pregnant Woman - Original Quality.mkv')
OUTPUT = Path('/content/drive/MyDrive/دبلجة/Billionaire_Granny_Arabic_Dub')

if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError("Google Drive is not mounted. Run the Drive authorization cell first.")
if not SOURCE.is_file():
    raise FileNotFoundError(f"Source video not found: {SOURCE}")
if SOURCE.stat().st_size < 1_000_000_000:
    raise RuntimeError(f"Source size is unexpectedly small ({SOURCE.stat().st_size:,} bytes). Stopping to avoid processing a partial file.")

mem = {}
for line in Path('/proc/meminfo').read_text().splitlines():
    key, _, value = line.partition(':')
    if value.strip().endswith('kB'):
        mem[key] = int(value.strip().split()[0]) / (1024 * 1024)
ram_total = mem.get('MemTotal', 0)
ram_available = mem.get('MemAvailable', 0)
disk = shutil.disk_usage('/content')
disk_free_gib = disk.free / (1024**3)

print("Source:", SOURCE.name)
print(f"Source size: {SOURCE.stat().st_size:,} bytes")
print(f"RAM total: {ram_total:.1f} GiB")
print(f"RAM available: {ram_available:.1f} GiB")
print(f"Free local disk: {disk_free_gib:.1f} GiB")
print("Output folder:", OUTPUT)
print("Existing output/checkpoints:", OUTPUT.exists())

if ram_total < 16:
    raise RuntimeError(
        f"Only {ram_total:.1f} GiB RAM is available. Stop here, open Runtime → Change runtime type, "
        "select High-RAM if available, then reconnect and run all cells again. No models have been loaded."
    )
if ram_available < 10:
    raise RuntimeError(
        f"Only {ram_available:.1f} GiB RAM is currently free. Restart the runtime and select High-RAM before continuing."
    )
if disk_free_gib < 16:
    raise RuntimeError(
        f"Only {disk_free_gib:.1f} GiB local disk is free; at least 16 GiB is required before downloading models."
    )

print("Preflight passed: memory and disk limits are satisfied.")

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path

REPO = Path('/content/AI-Creative-Studio')
TASK = REPO / 'scripts/dub_billionaire_granny_mounted.py'

def run_setup(cmd, label):
    print("\\n==", label, "==", flush=True)
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-12000:], flush=True)
    if result.returncode:
        raise RuntimeError(f"{label} failed (exit {result.returncode}). See output above.")

if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run_setup(['apt-get', 'update'], 'Update package index')
    run_setup(['apt-get', 'install', '-y', 'ffmpeg', 'git', 'build-essential', 'cmake', 'libsndfile1'],
              'Install FFmpeg and build prerequisites')

if not REPO.is_dir():
    run_setup(['git', 'clone', '--depth', '1', 'https://github.com/h776117073-eng/AI-Creative-Studio.git', str(REPO)],
              'Clone AI-Creative-Studio')
else:
    run_setup(['git', '-C', str(REPO), 'pull', '--ff-only'], 'Update AI-Creative-Studio')

if not TASK.is_file():
    raise FileNotFoundError(f"Colab dubbing runner is missing: {TASK}")

OUTPUT = Path('/content/drive/MyDrive/دبلجة/Billionaire_Granny_Arabic_Dub')
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'checkpoints').mkdir(parents=True, exist_ok=True)
print("\\nSetup passed. Input video stays in Drive; processing uses bounded, disk-backed chunks.")

## Start the dubbing
Before running this cell, confirm that the RAM preflight passed. This step can take a long time. It periodically backs up completed voice segments to Drive; do not delete the `checkpoints` folder while processing.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/AI-Creative-Studio')
TASK = REPO / 'scripts/dub_billionaire_granny_mounted.py'
os.environ['ASR_MODEL_INPUT'] = 'small'
os.environ['TRANSLATION_BACKEND_INPUT'] = 'google'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if not TASK.is_file():
    raise FileNotFoundError(TASK)

print("Starting resumable Arabic dubbing. Progress is continuously printed and mirrored to Drive.")
print("If the runtime is interrupted, run this cell again: valid checkpoints on Drive will be restored.")
process = subprocess.Popen(
    [sys.executable, str(TASK)],
    cwd=str(REPO),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors='replace',
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
exit_code = process.wait()
if exit_code != 0:
    raise RuntimeError(
        f"Dubbing stopped with exit code {exit_code}. Open the Drive folder "
        "'دبلجة/Billionaire_Granny_Arabic_Dub', inspect progress.json and progress.log, "
        "then rerun this cell to resume from verified checkpoints."
    )
print("The runner reports that MP4 validation and Drive-copy checksum verification passed.")

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown

OUT = Path('/content/drive/MyDrive/دبلجة/Billionaire_Granny_Arabic_Dub')
report_path = OUT / 'quality_report.json'
progress_path = OUT / 'progress.json'
final_path = OUT / 'Billionaire Granny Takes Home Fainted Pregnant Woman - Arabic Dub.mp4'

if progress_path.is_file():
    progress = json.loads(progress_path.read_text(encoding='utf-8'))
    print("Latest stage:", progress.get('stage', progress.get('status')))
    print("Updated UTC:", progress.get('updated_at_utc'))

if report_path.is_file():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps({
        "status": report.get("status"),
        "source_duration_seconds": report.get("source_duration_seconds"),
        "final_duration_seconds": report.get("final_duration_seconds"),
        "speaker_count": report.get("speaker_count"),
        "segment_count": report.get("segment_count"),
        "emotion_tags": report.get("emotion_tags"),
        "final_size_bytes": report.get("final_size_bytes"),
        "final_sha256": report.get("final_sha256"),
        "full_ffmpeg_decode": report.get("full_ffmpeg_decode"),
        "drive_copy_sha256_check": report.get("drive_copy_sha256_check"),
        "visual_lip_sync": report.get("visual_lip_sync"),
    }, ensure_ascii=False, indent=2))
else:
    print("No final quality report exists yet. The work may be incomplete; read progress.json and progress.log before rerunning.")

print("\\nDrive output folder:")
print("https://drive.google.com/drive/my-drive")
print("Open: دبلجة / Billionaire_Granny_Arabic_Dub")
if final_path.is_file():
    print("\\nFinal MP4 exists in the mounted Drive and is ready to open.")
else:
    print("\\nFinal MP4 has not yet been verified. Do not treat this run as complete.")